# 05 — Baselines, Model Training and Hyperparameter Tuning (Stages 6–8)

* **Baselines** (no learning): previous day, same day last week, 7-day moving average.
* **ML models:** Linear Regression, Random Forest, HistGradientBoosting, XGBoost (scikit-learn pipelines).
* **Tuning:** `RandomizedSearchCV` with **rolling-origin folds built from training dates only**.
* **Selection:** lowest validation MAE. The test period is used once, at the end.

Takes about 5 minutes on a laptop.

> **Colab:** run the setup cell; it asks for the project ZIP the first time. Put `Online Retail.xlsx` in `data/raw/` (or run notebook 01 first, which uploads it).

In [1]:
import sys, os, zipfile, io
from pathlib import Path
IN_COLAB = "google.colab" in sys.modules
ROOT = Path.cwd()
for c in (ROOT, ROOT.parent, ROOT / "AI-Sales-Forecasting-Inventory"):
    if (c / "ml").is_dir():
        ROOT = c
        break
if IN_COLAB and not (ROOT / "ml").is_dir():
    from google.colab import files
    print("Upload the project ZIP once (AI-Sales-Forecasting-Inventory.zip):")
    for name, data in files.upload().items():
        zipfile.ZipFile(io.BytesIO(data)).extractall("/content")
    ROOT = Path("/content/AI-Sales-Forecasting-Inventory")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ["SFIO_PROJECT_ROOT"] = str(ROOT)

import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from IPython.display import Image, display, Markdown
from ml import config
config.ensure_directories()
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 180)
sns.set_theme(style="whitegrid")
print("Project root:", ROOT.name, "| Colab:", IN_COLAB)

Project root: AI-Sales-Forecasting-Inventory | Colab: False


In [2]:
from ml import train, feature_engineering as fe
features = pd.read_parquet(config.PROCESSED_DATA_DIR / "features.parquet")
print(features["split"].value_counts())

split
train         96328
validation    21472
test          20900
Name: count, dtype: int64


## Rolling-origin folds used for tuning

In [3]:
tr = features[features.split == "train"].reset_index(drop=True)
for i, (a, b) in enumerate(fe.rolling_origin_folds(tr["Date"])):
    print(f"fold {i+1}: train up to {tr['Date'].iloc[a].max().date()} | validate {tr['Date'].iloc[b].min().date()} to {tr['Date'].iloc[b].max().date()}")

fold 1: train up to 2011-03-02 | validate 2011-03-03 to 2011-05-05
fold 2: train up to 2011-05-05 | validate 2011-05-06 to 2011-07-04
fold 3: train up to 2011-07-04 | validate 2011-07-05 to 2011-08-31


In [4]:
result = train.train_and_evaluate(features, progress=print)

Loss-function design check on validation data


Tuning linear_regression (no hyperparameters)


linear_regression: validation MAE 19.312 (0.3s)
Tuning random_forest (6 candidates x 3 folds)


random_forest: validation MAE 20.389 (271.6s)
Tuning hist_gradient_boosting (12 candidates x 3 folds)


hist_gradient_boosting: validation MAE 18.424 (190.8s)
Tuning xgboost (12 candidates x 3 folds)


xgboost: validation MAE 18.490 (56.0s)
Selected by validation MAE: hist_gradient_boosting
Refitting linear_regression on train+validation for the test evaluation


Refitting random_forest on train+validation for the test evaluation


Refitting hist_gradient_boosting on train+validation for the test evaluation


Refitting xgboost on train+validation for the test evaluation


Refitting hist_gradient_boosting on all data for live forecasting


## Loss-function design check (validation only)
Absolute-error loss gets a low MAE but under-forecasts total demand badly, which is dangerous for inventory, so it is excluded from tuning.

In [5]:
pd.DataFrame(result["loss_experiment"])

,loss,validation_mae,validation_wape_pct,validation_bias,bias_pct_of_mean_demand
0,squared_error,19.534725,91.097366,-4.620118,-21.545251
1,poisson,18.558816,86.546355,-6.807153,-31.744171
2,absolute_error,17.726547,82.665187,-13.239872,-61.742227


## Comparison (validation and test)

In [6]:
result["comparison"].round(3)

,model,type,split,n,mae,rmse,wape_pct,mape_pct,mape_coverage_pct,r2,bias
0,baseline_previous_day,baseline,validation,21472,25.428,76.485,118.582,127.124,52.967,-0.576,-3.700
1,baseline_previous_day,baseline,test,20900,26.634,95.656,111.657,121.611,58.373,-0.595,-3.382
2,baseline_same_day_last_week,baseline,validation,21472,26.044,78.855,121.452,122.978,52.967,-0.675,-1.325
3,baseline_same_day_last_week,baseline,test,20900,26.254,91.155,110.063,126.254,58.373,-0.449,-0.757
4,baseline_moving_average_7,baseline,validation,21472,21.152,60.485,98.638,89.963,52.967,0.015,-3.940
5,baseline_moving_average_7,baseline,test,20900,21.226,71.675,88.987,88.690,58.373,0.104,-3.646
6,linear_regression,ml,validation,21472,19.312,57.368,90.058,76.637,52.967,0.114,-3.299
7,random_forest,ml,validation,21472,20.389,57.286,95.082,82.450,52.967,0.116,-0.415
8,hist_gradient_boosting,ml,validation,21472,18.424,58.527,85.918,63.499,52.967,0.077,-7.904
9,xgboost,ml,validation,21472,18.490,57.685,86.225,66.627,52.967,0.104,-5.652


In [7]:
print("Selected (by validation MAE):", result["best_model_name"])
print("Best baseline (validation):", result["best_baseline_name"])
pd.Series(train.improvement_summary(result["comparison"], result["best_model_name"], result["best_baseline_name"]))

Selected (by validation MAE): hist_gradient_boosting
Best baseline (validation): baseline_moving_average_7


best_model                hist_gradient_boosting
best_baseline          baseline_moving_average_7
test_mae_model                         21.274879
test_mae_baseline                      21.226459
mae_improvement_pct                    -0.228111
test_wape_model                        89.189965
test_wape_baseline                     88.986976
beats_baseline                             False
dtype: object

In [8]:
pd.DataFrame({m: v["best_params"] for m, v in result["tuning"].items()}).T

,model__n_estimators,model__min_samples_split,model__min_samples_leaf,model__max_features,model__max_depth,model__l2_regularization,model__learning_rate,model__loss,model__max_iter,model__max_leaf_nodes,model__colsample_bytree,model__objective,model__subsample,model__tweedie_variance_power
linear_regression,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
random_forest,300.0,20.0,3.0,0.5,8.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
hist_gradient_boosting,NaN,NaN,NaN,NaN,NaN,1.0,0.050222,poisson,400,31,NaN,NaN,NaN,NaN
xgboost,199,NaN,NaN,NaN,8,NaN,0.022862,NaN,NaN,NaN,0.577997,reg:tweedie,0.608234,1.35


## Save model artifacts (Joblib + JSON)

In [9]:
fes = fe.build_dataset(pd.read_parquet(config.PROCESSED_DATA_DIR / "cleaned_sales.parquet"))
meta = train.save_training_outputs(result, fes["split_info"], fes["products"], fe.closed_weekdays(fes["calendar"]),
                                   fes["calendar"]["Date"].max(), len(features))
print("Saved artifacts/demand_model.joblib and model_metadata.json for", meta["model_name"])

Saved artifacts/demand_model.joblib and model_metadata.json for hist_gradient_boosting
